In [3]:
!pip install --upgrade torchao

In [4]:
import sys
import os
import torch
from google.colab import drive
from tqdm import tqdm
from transformers import AutoTokenizer
from peft import PeftModel
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import warnings

# Disattiviamo i warning di divisione per zero per le classi mai predette dal modello base
warnings.filterwarnings('ignore')

# Montaggio Drive e Path
drive.mount('/content/drive')
BASE_DRIVE = '/content/drive/MyDrive/DeepLearning'
SRC_DIR = os.path.join(BASE_DRIVE, 'src')
if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)

from dataset import CLEVRDataset
from models import MultimodalCoT
from torchvision import transforms
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained("t5-base")

transform = transforms.Compose([
    transforms.Resize((384, 384)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

TEST_INDEX = os.path.join(BASE_DRIVE, 'indexes/val_index.json')
TEST_IMG_DIR = os.path.join(BASE_DRIVE, 'data/processed/images/val')

test_dataset = CLEVRDataset(
    index_path=TEST_INDEX,
    img_dir=TEST_IMG_DIR,
    tokenizer=tokenizer,
    transform=transform,
    stage='stage1'
)

def custom_collate_fn(batch):
    return {
        'pixel_values': torch.stack([item['pixel_values'] for item in batch]),
        'input_text': [item['input_text'] for item in batch],
        'target_text': [item['target_text'] for item in batch],
        'raw_item': [item['raw_item'] for item in batch]
    }

test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, collate_fn=custom_collate_fn)
print(f"✅ Test Set pronto. Campioni: {len(test_dataset)}")

Mounted at /content/drive


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

✅ Test Set pronto. Campioni: 1000


In [5]:
print("🔄 Inizializzazione dell'architettura e caricamento dei pesi storici...")

# RICARICHIAMO IL MODELLO PULITO PER ELIMINARE IL VECCHIO WRAPPER
model = MultimodalCoT().to(device)

# 1. Salviamo i pesi visivi CASUALI (Modello Non Addestrato)
import copy
base_projector = copy.deepcopy(model.projector.state_dict())
base_cross_attn = copy.deepcopy(model.cross_attention.state_dict())

# 2. Carichiamo i pesi visivi dello STAGE 1
stage1_vis = torch.load(os.path.join(BASE_DRIVE, 'lora_adapter_final', 'custom_modules.pth'))
stage1_proj = stage1_vis['projector']
stage1_attn = stage1_vis['cross_attention']

# 3. Carichiamo i pesi visivi dello STAGE 2
stage2_vis = torch.load(os.path.join(BASE_DRIVE, 'lora_adapter_stage2', 'custom_modules.pth'))
stage2_proj = stage2_vis['projector']
stage2_attn = stage2_vis['cross_attention']

# 4. Gestione NATIVA degli Adattatori LoRA (Fix per il ValueError)
# Carichiamo gli adattatori sfruttando l'integrazione nativa di Transformers
model.llm.load_adapter(os.path.join(BASE_DRIVE, 'lora_adapter_final'), adapter_name="stage1")
model.llm.load_adapter(os.path.join(BASE_DRIVE, 'lora_adapter_stage2'), adapter_name="stage2")

# Funzioni di Switch Rapido
def set_model_state(state):
    if state == "base":
        model.llm.disable_adapters()
        model.projector.load_state_dict(base_projector)
        model.cross_attention.load_state_dict(base_cross_attn)
    elif state == "stage1":
        model.llm.enable_adapters()
        model.llm.set_adapter("stage1")
        model.projector.load_state_dict(stage1_proj)
        model.cross_attention.load_state_dict(stage1_attn)
    elif state == "stage2":
        model.llm.enable_adapters()
        model.llm.set_adapter("stage2")
        model.projector.load_state_dict(stage2_proj)
        model.cross_attention.load_state_dict(stage2_attn)

model.eval()
print("✅ Sistema multi-stato pronto per il benchmark.")

🔄 Inizializzazione dell'architettura e caricamento dei pesi storici...


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.23GB            

model.safetensors: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  892MB            

model.safetensors: downloading bytes:           |  0.00B            

...

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/144 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/144 [00:00<?, ?it/s]

[transformers] T5ForConditionalGeneration LOAD REPORT from: /content/drive/MyDrive/DeepLearning/lora_adapter_stage2
Key                                                                   | Status  | 
----------------------------------------------------------------------+---------+-
encoder.block.{0...11}.layer.0.SelfAttention.q.lora_B.stage1.weight   | MISSING | 
encoder.block.{0...11}.layer.0.SelfAttention.v.lora_B.stage1.weight   | MISSING | 
decoder.block.{0...11}.layer.0.SelfAttention.v.lora_B.stage1.weight   | MISSING | 
decoder.block.{0...11}.layer.0.SelfAttention.v.lora_A.stage1.weight   | MISSING | 
decoder.block.{0...11}.layer.1.EncDecAttention.v.lora_B.stage1.weight | MISSING | 
encoder.block.{0...11}.layer.0.SelfAttention.v.lora_A.stage1.weight   | MISSING | 
encoder.block.{0...11}.layer.0.SelfAttention.q.lora_A.stage1.weight   | MISSING | 
decoder.block.{0...11}.layer.1.EncDecAttention.v.lora_A.stage1.weight | MISSING | 
decoder.block.{0...11}.layer.0.SelfAttention.q.lora_A.

✅ Sistema multi-stato pronto per il benchmark.


In [6]:
def evaluate_architecture(mode="base", max_samples=None):
    correct_rationales = 0
    total_samples = 0

    all_true_answers = []
    all_pred_answers = []

    progress_bar = tqdm(test_loader, desc=f"Valutazione: {mode.upper()}")

    with torch.no_grad():
        for batch in progress_bar:
            pixel_values = batch['pixel_values'].to(device)
            questions = [item['question'] for item in batch['raw_item']]
            true_answers = [str(item.get('answer', '')).strip().lower() for item in batch['raw_item']]

            # Recuperiamo la stringa target completa per estrarre la verità
            target_texts = batch['target_text']

            # --- LOGICA FASE 1 ---
            if mode == "base":
                set_model_state("base")
            else:
                set_model_state("stage1")

            inputs_s1 = tokenizer([f"Question: {q} Answer:" for q in questions], return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)

            with torch.amp.autocast('cuda'):
                gen_s1 = model.generate(
                    input_ids=inputs_s1.input_ids,
                    attention_mask=inputs_s1.attention_mask,
                    pixel_values=pixel_values,
                    max_new_tokens=256 # 💥 FIX: Aumentato a 256 per evitare il troncamento del CoT!
                )[0]

            outputs_s1 = tokenizer.batch_decode(gen_s1, skip_special_tokens=True)

            # 💥 FIX: Parsing sicuro in caso di modelli ribelli o troncamenti residui
            pred_rationales = []
            pred_answers_s1 = []
            for out in outputs_s1:
                if "The final answer is:" in out:
                    parts = out.split("The final answer is:")
                    pred_rationales.append(parts[0].replace("Let's think step by step.", "").strip())
                    pred_answers_s1.append(parts[1].strip().lower())
                else:
                    pred_rationales.append(out.replace("Let's think step by step.", "").strip())
                    pred_answers_s1.append("")

            # --- LOGICA FASE 2 E RISPOSTE FINALI ---
            final_predictions = []

            if mode in ["base", "stage1"]:
                final_predictions = pred_answers_s1

            elif mode == "stage2":
                set_model_state("stage2")
                inputs_s2_text = [f"Question: {q} Rationale: {r} Answer:" for q, r in zip(questions, pred_rationales)]
                inputs_s2 = tokenizer(inputs_s2_text, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)

                with torch.amp.autocast('cuda'):
                    gen_s2 = model.generate(
                        input_ids=inputs_s2.input_ids,
                        attention_mask=inputs_s2.attention_mask,
                        pixel_values=pixel_values,
                        max_new_tokens=10
                    )[0]

                final_predictions = [ans.strip().lower() for ans in tokenizer.batch_decode(gen_s2, skip_special_tokens=True)]

            all_true_answers.extend(true_answers)
            all_pred_answers.extend(final_predictions)

            # 💥 FIX: Estrazione del VERO rationale dividendo il target_text originale
            for i in range(len(true_answers)):
                full_target = target_texts[i]
                if "The final answer is:" in full_target:
                    true_rationale = full_target.split("The final answer is:")[0].replace("Let's think step by step.", "").strip()
                else:
                    true_rationale = full_target.replace("Let's think step by step.", "").strip()

                true_words = set(true_rationale.split())
                pred_words = set(pred_rationales[i].split())
                if len(true_words) > 0:
                    overlap = len(true_words.intersection(pred_words)) / len(true_words)
                    if overlap > 0.8: # Consideriamo corretto l'80% di overlap testuale
                        correct_rationales += 1

                total_samples += 1

            current_acc = accuracy_score(all_true_answers, all_pred_answers)
            progress_bar.set_postfix({'Ans_Acc': f"{current_acc*100:.1f}%"})

            if max_samples and total_samples >= max_samples:
                break

    # --- CALCOLO METRICHE GLOBALI (MACRO) ---
    ans_acc = accuracy_score(all_true_answers, all_pred_answers) * 100
    precision, recall, f1, _ = precision_recall_fscore_support(all_true_answers, all_pred_answers, average='macro', zero_division=0)
    rat_acc = (correct_rationales / total_samples) * 100

    return rat_acc, ans_acc, precision * 100, recall * 100, f1 * 100

In [7]:
MAX_SAMPLES = None

print("INIZIO BENCHMARK SUL TEST SET\n" + "="*95)

# 1. Test Modello Vergine
rat_base, ans_base, p_base, r_base, f1_base = evaluate_architecture("base", max_samples=MAX_SAMPLES)

# 2. Test Modello a 1 Fase (Sintomi del Language Bias)
rat_s1, ans_s1, p_s1, r_s1, f1_s1 = evaluate_architecture("stage1", max_samples=MAX_SAMPLES)

# 3. Test Modello a 2 Fasi (Cascade Reasoning risolutivo)
rat_s2, ans_s2, p_s2, r_s2, f1_s2 = evaluate_architecture("stage2", max_samples=MAX_SAMPLES)

print("\n" + "="*95)
print("📊 RISULTATI FINALI DEL BENCHMARK SUL TEST SET")
print("="*95)
print(f"{'Architettura':<25} | {'Acc. Rationale':<14} | {'Acc. Risposta':<13} | {'Precision':<9} | {'Recall':<9} | {'F1-Score':<9}")
print("-" * 95)
print(f"{'1. Non Addestrata (Base)':<25} | {rat_base:>13.2f}% | {ans_base:>12.2f}% | {p_base:>8.2f}% | {r_base:>8.2f}% | {f1_base:>8.2f}%")
print(f"{'2. Singola Fase (Stage 1)':<25} | {rat_s1:>13.2f}% | {ans_s1:>12.2f}% | {p_s1:>8.2f}% | {r_s1:>8.2f}% | {f1_s1:>8.2f}%")
print(f"{'3. Due Fasi (Cascade)':<25} | {rat_s2:>13.2f}% | {ans_s2:>12.2f}% | {p_s2:>8.2f}% | {r_s2:>8.2f}% | {f1_s2:>8.2f}%")
print("="*95)

INIZIO BENCHMARK SUL TEST SET


Valutazione: STAGE2: 100%|██████████| 63/63 [05:16<00:00,  5.02s/it, Ans_Acc=46.1%]


📊 RISULTATI FINALI DEL BENCHMARK SUL TEST SET
Architettura              | Acc. Rationale | Acc. Risposta | Precision | Recall    | F1-Score 
-----------------------------------------------------------------------------------------------
1. Non Addestrata (Base)  |          0.00% |         0.00% |     0.00% |     0.00% |     0.00%
2. Singola Fase (Stage 1) |        100.00% |        48.10% |    28.42% |    28.09% |    22.56%
3. Due Fasi (Cascade)     |        100.00% |        46.10% |    30.93% |    31.34% |    28.30%
